# Variáveis

**Objetivo:** criar informações sobre o histórico de cada cartão e medir se elas ajudam.

**Regras:**
- Cada transação só enxerga transações de instantes anteriores.
- `isFraud` não entra nas agregações, porque o rótulo só é conhecido depois do chargeback.
- Treino só com os dados de treino e resultados só na validação. O teste não entra em nenhum cálculo.

**Decisões:** D11 (colunas C) e D12 (agregações por cartão), em `docs/decisoes.md`.

In [1]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
from divisao_temporal import calcular_cortes, dividir_por_tempo
from metricas import selecionar_revisadas, valor_capturado
from variaveis import agregacoes_anteriores, chave_de_cartao

ARQUIVO = RAIZ / "data" / "processed" / "treino.parquet"
nomes = pq.ParquetFile(ARQUIVO).schema_arrow.names
COLUNAS_C = [n for n in nomes if re.fullmatch(r"C\d+", n)]
COLUNAS_ID = [n for n in nomes if re.fullmatch(r"id_\d+", n)]
COLUNAS_CATEGORIA = ["ProductCD", "card4", "card6"]
COLUNAS_CARTAO = [f"card{i}" for i in range(1, 7)]

tabela = pd.read_parquet(
    ARQUIVO,
    columns=[
        "TransactionDT", "isFraud", "TransactionAmt", "addr1",
        *dict.fromkeys([*COLUNAS_CATEGORIA, *COLUNAS_CARTAO]), *COLUNAS_C, *COLUNAS_ID,
    ],
)
tabela["tem_identidade"] = tabela[COLUNAS_ID].notna().any(axis=1).astype("int8")
tabela = tabela.drop(columns=COLUNAS_ID)

corte_treino, corte_validacao = calcular_cortes(tabela["TransactionDT"])
treino, validacao, _ = dividir_por_tempo(tabela, corte_treino, corte_validacao)
del _
tabela = pd.concat([treino, validacao])  # daqui em diante, sem o teste
del treino, validacao
print(f"Linhas sem o teste: {len(tabela):,}")

Linhas sem o teste: 472,432


## 1. O que é "um cartão"?

A base não informa um identificador de cartão. Aqui comparamos três aproximações. "Máximo de transações" mostra o maior grupo: quanto maior, mais provável que ele misture clientes diferentes.

In [2]:
OPCOES_DE_CHAVE = {
    "card1": ["card1"],
    "card1 a card6": COLUNAS_CARTAO,
    "card1 a card6 + addr1": [*COLUNAS_CARTAO, "addr1"],
}

agregacoes = {}
linhas = []
for nome, colunas in OPCOES_DE_CHAVE.items():
    chave = chave_de_cartao(tabela, colunas)
    agregacoes[nome] = agregacoes_anteriores(tabela["TransactionDT"], tabela["TransactionAmt"], chave)
    contagem = chave.value_counts()
    linhas.append(
        {
            "Chave": nome,
            "Grupos": len(contagem),
            "Mediana de transações": float(contagem.median()),
            "Máximo de transações": int(contagem.max()),
            "% das transações sem histórico": round(100 * float((agregacoes[nome]["qtd_anteriores"] == 0).mean()), 2),
        }
    )
pd.DataFrame(linhas).set_index("Chave")

,Grupos,Mediana de transações,Máximo de transações,% das transações sem histórico
Chave,,,,
card1,12730,4.0,12205,2.69
card1 a card6,13792,3.0,11109,2.92
card1 a card6 + addr1,39087,2.0,8081,8.27


## 2. As agregações ajudam?

Mesma regressão logística dos baselines, mudando só as colunas. Quanto maior a PR-AUC e o valor capturado, melhor. Quanto menor o Brier, melhor.

In [3]:
separa_treino = tabela["TransactionDT"] <= corte_treino
rotulos = tabela.loc[~separa_treino, "isFraud"].astype("int8")
valores = tabela.loc[~separa_treino, "TransactionAmt"].astype("float64")


def preparar(agregadas: pd.DataFrame | None) -> pd.DataFrame:
    x = np.log1p(tabela[COLUNAS_C].astype("float64"))
    x["log_valor"] = np.log1p(tabela["TransactionAmt"].astype("float64"))
    x["tem_identidade"] = tabela["tem_identidade"]
    for coluna in COLUNAS_CATEGORIA:
        x[coluna] = tabela[coluna].astype("object").fillna("(vazio)")
    if agregadas is not None:
        for coluna in agregadas.columns:
            # Sem histórico, o valor vira 0 e a coluna sem_historico marca o caso.
            x[coluna] = np.log1p(agregadas[coluna].fillna(0).astype("float64"))
        x["sem_historico"] = (agregadas["qtd_anteriores"] == 0).astype("int8")
    return x


def avaliar(nome: str, agregadas: pd.DataFrame | None, com_c: bool) -> tuple[dict, pd.Series]:
    x = preparar(agregadas)
    numericas = ["log_valor", "tem_identidade", *(COLUNAS_C if com_c else [])]
    if agregadas is not None:
        numericas += [*agregadas.columns, "sem_historico"]
    modelo = Pipeline(
        [
            (
                "preparo",
                ColumnTransformer(
                    [
                        ("numericas", StandardScaler(), numericas),
                        ("categorias", OneHotEncoder(handle_unknown="ignore"), COLUNAS_CATEGORIA),
                    ]
                ),
            ),
            ("logistica", LogisticRegression(max_iter=1000)),
        ]
    )
    modelo.fit(x[separa_treino], tabela.loc[separa_treino, "isFraud"])
    probabilidade = pd.Series(modelo.predict_proba(x[~separa_treino])[:, 1], index=rotulos.index)
    linha = {
        "Modelo": nome,
        "PR-AUC": average_precision_score(rotulos, probabilidade),
        "Brier": brier_score_loss(rotulos, probabilidade),
    }
    for capacidade in [0.01, 0.03]:
        revisadas = selecionar_revisadas(probabilidade, capacidade)
        rotulo = f"{100 * capacidade:.0f}%"
        linha[f"Precisão {rotulo}"] = 100 * rotulos.loc[revisadas].mean()
        linha[f"Valor capturado {rotulo}"] = valor_capturado(rotulos, probabilidade, valores, capacidade)
    return linha, probabilidade


resultados = []
for nome, agregadas, com_c in [
    ("Básicas", None, False),
    ("Básicas + C", None, True),
    *[(f"Básicas + {chave}", agregacoes[chave], False) for chave in OPCOES_DE_CHAVE],
    *[(f"Básicas + C + {chave}", agregacoes[chave], True) for chave in OPCOES_DE_CHAVE],
]:
    linha, _ = avaliar(nome, agregadas, com_c)
    resultados.append(linha)
pd.DataFrame(resultados).set_index("Modelo").round(4)

,PR-AUC,Brier,Precisão 1%,Valor capturado 1%,Precisão 3%,Valor capturado 3%
Modelo,,,,,,
Básicas,0.1351,0.0356,25.0423,4.1666,20.1749,6.4651
Básicas + C,0.4357,0.0278,86.2944,11.3900,51.3262,26.6538
Básicas + card1,0.1361,0.0355,23.9425,2.9426,21.2754,6.8764
Básicas + card1 a card6,0.1327,0.0356,22.6734,2.8636,20.5982,6.7065
Básicas + card1 a card6 + addr1,0.1376,0.0355,24.1117,2.8731,21.4165,6.7166
Básicas + C + card1,0.4319,0.0279,85.2792,10.8931,50.3668,26.3281
Básicas + C + card1 a card6,0.4303,0.0280,85.4484,10.9227,49.9153,25.8851
Básicas + C + card1 a card6 + addr1,0.4337,0.0278,85.2792,10.7785,50.9876,25.6481


## 3. Checagem das colunas C (D11)

As colunas C são contagens calculadas pela Vesta. Se uma coluna contar só o passado, o valor dela para um mesmo cartão tende a começar baixo e nunca diminuir. Se contar o período inteiro, inclusive o futuro, tende a ficar igual em todas as transações do cartão.

A checagem usa só o período de treino e a chave `card1` a `card6` + `addr1`, que tem os menores grupos. Entram só as chaves com pelo menos 5 transações. Como a chave não identifica um cartão real e o cartão pode ter histórico anterior ao início da base, o resultado é só um indício.

In [4]:
no_treino = tabela["TransactionDT"] <= corte_treino
chave_checagem = chave_de_cartao(tabela, [*COLUNAS_CARTAO, "addr1"])
dados_c = (
    tabela.loc[no_treino, ["TransactionDT", *COLUNAS_C]]
    .assign(chave=chave_checagem[no_treino])
    .sort_values(["chave", "TransactionDT"], kind="stable")
)
tamanho = dados_c.groupby("chave").size()
dados_c = dados_c[dados_c["chave"].isin(tamanho[tamanho >= 5].index)]
print(f"Chaves analisadas: {dados_c['chave'].nunique():,} | Transações: {len(dados_c):,}")

linhas_c = []
for coluna in COLUNAS_C:
    por_chave = (
        dados_c.assign(diferenca=dados_c.groupby("chave")[coluna].diff())
        .groupby("chave")
        .agg(
            menor_diferenca=("diferenca", "min"),
            valores_distintos=(coluna, "nunique"),
            primeiro=(coluna, "first"),
            ultimo=(coluna, "last"),
        )
    )
    linhas_c.append(
        {
            "Coluna": coluna,
            "% das chaves em que nunca diminui": 100 * (por_chave["menor_diferenca"] >= 0).mean(),
            "% das chaves em que é sempre igual": 100 * (por_chave["valores_distintos"] == 1).mean(),
            "Mediana na 1ª transação": por_chave["primeiro"].median(),
            "Mediana na última transação": por_chave["ultimo"].median(),
        }
    )
pd.DataFrame(linhas_c).set_index("Coluna").round(1)

Chaves analisadas: 9,908 | Transações: 312,608


,% das chaves em que nunca diminui,% das chaves em que é sempre igual,Mediana na 1ª transação,Mediana na última transação
Coluna,,,,
C1,23.3,15.4,1.0,1.0
C2,24.5,14.6,1.0,1.0
C3,94.5,94.3,0.0,0.0
C4,52.1,49.2,0.0,0.0
C5,34.4,28.8,0.0,0.0
C6,25.1,16.8,1.0,1.0
C7,95.6,95.0,0.0,0.0
C8,48.2,45.3,0.0,0.0
C9,27.9,19.2,1.0,1.0


## Conclusões das variáveis

- Nenhuma chave identifica um cartão real. A chave `card1` a `card6` + `addr1` é a mais próxima: tem 39.087 grupos e o maior deles tem 8.081 transações, contra 12.205 em `card1`. Em troca, 8,27% das transações ficam sem histórico.
- O "sem histórico" baixo de `card1` (2,69%) é, em parte, sintoma da mistura de clientes: quase toda transação encontra compras anteriores, mas de outras pessoas.
- Na regressão logística com colunas C, as agregações quase não mudaram o resultado. A PR-AUC foi de 0,4357 sem agregações para valores entre 0,4303 e 0,4337 com elas, e o valor capturado em 3% caiu até 1,00 ponto. Com uma única validação, essas diferenças não se distinguem de variação normal.
- A regressão logística só capta relações lineares. Por isso, as agregações serão testadas também no LightGBM (D12).
- A checagem das colunas C não foi conclusiva. Em C1, o valor nunca diminui em 23,3% das chaves e é sempre igual em 15,4%; em C13, os números são 13,9% e 2,4%. Isso não parece nem um contador puro do passado nem uma contagem do período inteiro. A mistura de clientes na chave ou uma janela de contagem podem explicar o padrão. Por isso, o modelo será treinado com e sem as colunas C (D11).
- Estes resultados valem só para a validação, e o teste não foi usado.